# Onboarding a lab: automated and shared data pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SCENE-Collaboration/SCENE-pipelines/blob/dev/examples/tutorials/03_scene_onboarding_demo.ipynb)

How a lab puts its pipeline on SCENE, end to end, with real DeepLabCut data: a mouse, two sessions months
apart, the shared tables (`Session`, `VideoRecording`, `dlc.Pose`, …), the lab's own tables on top, and
analyses that compute themselves. The lab's code is the package
[`scene_onboarding_demo`](https://github.com/SCENE-Collaboration/SCENE-pipelines/tree/dev/examples/tutorials/scene_onboarding_demo), next to this
notebook.

Run [Getting started](01_getting_started.ipynb) first for the basics of the shared tables.

**About the data**

The mouse *DemoTuatara* and every metadata value in the data's `sessions/` folder are made up for
this demo. The recordings and DeepLabCut results are real data from unpublished neural recordings in the AR task.
If you use this data, please cite

> Benquet, C., Sainsbury, T., Bruneau, L., et al., & Mathis, M. W. (2026). Visual uncertainty and task demands shape active sensing strategies in mice. *Current Biology*, *36*(14), 3523–3534.e9. [https://doi.org/10.1016/j.cub.2026.06.011](https://doi.org/10.1016/j.cub.2026.06.011)

**Before you start** (on Colab, the first code cell does all of this)

- Install the package with its DeepLabCut and video extras, from the repository root:
  `pip install -e ".[dlc,video]" matplotlib`
- Start the local MySQL database: `make init` once, then `make db_up`
- Run this notebook from `examples/tutorials/`, so that `tutorial_utils` and `scene_onboarding_demo` can be
  imported

Everything is written to databases named `tutorial_onboarding_*`. Every step can run again.

In [ ]:
# On Google Colab only: a local MySQL server, the package and this folder. Elsewhere this cell does nothing.
import os
import sys

SCENE_REF = "dev"  # the branch or tag of SCENE-pipelines to install

if "google.colab" in sys.modules:
    !apt-get -qq update > /dev/null && apt-get -qq install -y mysql-server > /dev/null
    !service mysql start
    !mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'tutorial'"
    !pip install -q "scene-pipelines[dlc,video] @ git+https://github.com/SCENE-Collaboration/SCENE-pipelines.git@{SCENE_REF}"
    !git clone -q --depth 1 -b {SCENE_REF} https://github.com/SCENE-Collaboration/SCENE-pipelines.git /content/SCENE-pipelines
    %cd /content/SCENE-pipelines/examples/tutorials
    os.environ["DJ_PASS"] = "tutorial"

## The data

The lab's data is one zip of about 54 MB: its session files (`sessions/`), a DeepLabCut project and the
recordings. This cell downloads it and unpacks it into `scene_onboarding_data/`, next to this notebook;
`layout.DATA` names that folder.

- Already have the zip? Put it next to this notebook, and the cell only unpacks it.
- Want the data elsewhere? Change `DATA` in `scene_onboarding_demo/layout.py`.
- Stopped halfway? Delete `scene_onboarding_data/` and run the cell again.

In [ ]:
import shutil
import urllib.request
from pathlib import Path

from scene_onboarding_demo.layout import DATA

URL = "https://github.com/SCENE-Collaboration/SCENE-pipelines/releases/download/tutorial-data/scene-onboarding-demo-data-v1.zip"
ZIP = Path("scene-onboarding-demo-data-v1.zip")

if not DATA.exists():
    if not ZIP.exists():
        urllib.request.urlretrieve(URL, ZIP)
    shutil.unpack_archive(ZIP, DATA)
sorted(path.name for path in DATA.iterdir())

## Connect and set up

`connect()` connects like the other tutorials, with the prefix `tutorial_onboarding_`, and binds the
shared schemas and the lab's own.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from scene_onboarding_demo.db import connect

connect()

Once per lab, an admin registers the lab, its project, its experimenters' codes and its cameras.

In [ ]:
from scene_onboarding_demo.register import setup_lab

setup_lab()

## How this lab organises its data

`layout.py` is the one place that describes the lab's data: where it lives and how a session folder is
named. Everything below reads it from there instead of assuming a structure.

Codes in the shared tables are seen by the whole consortium, so they must not name anyone. DeepLabCut names
a project folder after its labeller; `DLC_PROJECT_CODES` gives each project a code of the lab's own.

In [ ]:
from scene_onboarding_demo import layout

print(layout.__doc__)
print("lab:     ", layout.LAB)
print("project: ", layout.PROJECT)
print("cameras: ", layout.CAMERAS)
print("DeepLabCut projects:", layout.DLC_PROJECT_CODES)

## One automated call: `ingest()`

Subject → Session → video and poses → computed results, for one session folder:

In [ ]:
from scene_onboarding_demo.display import display_code
from scene_onboarding_demo.ingest import ingest

display_code(ingest)

## Step by step

### 1. Validated data discovery in Python

In [ ]:
from scene_onboarding_demo.layout import discover_sessions
from scene_onboarding_demo.manifest import read_session_folder

# Discover all the sessions on disk
sessions = discover_sessions()
for folder in sessions:
    print(folder.name)

FIRST, SECOND = sessions

# Inspect a specific session
manifest = read_session_folder(FIRST)
pd.Series(manifest.session.model_dump(), name=manifest.session.session_code).to_frame()

A wrong field fails here, by name, and nothing is written. Uncomment to try:

In [ ]:
# from scene_onboarding_demo.manifest import SessionEntry, read_json

# broken = {**read_json(layout.gui_file(FIRST)), "weight_percentage": "heavy", "rig": 2}

# SessionEntry(**broken)

## 2. Ingesting into the database

### 2a. The SCENE shared layer

The mouse and the session, through the helpers of `scene_pipelines.ingestion`:

In [ ]:
from scene_pipelines.schemas.mouse.mouse import Mouse
from scene_pipelines.schemas.scene.session import Session
from scene_pipelines.schemas.scene.subject import Subject

from scene_onboarding_demo.register import register_mouse, register_session

mouse = register_mouse(manifest.mouse)
session = register_session(manifest.session)

display((Subject * Mouse & mouse).proj("subject_code", "sex", "date_of_birth", "strain_name"))
Session & session

The DeepLabCut project, in the shared `dlc` schema: `DlcProject` under the lab's code, and
`ProjectConfig`, one row per version of its `config.yaml`, with the keypoints and the likelihood threshold
(`pcutoff`). The file itself is not uploaded: SCENE records its hash.

The project's labelled frames (`CollectedData_*.h5`) are not in the shared schema yet, so this version of
the demo does not register them.

In [ ]:
from scene_pipelines.schemas.dlc.project import DlcProject, ProjectConfig

from scene_onboarding_demo.register import register_dlc_project

config = register_dlc_project(layout.DATA / manifest.recording.dlc_project)
display(DlcProject & config)
(ProjectConfig & config).proj("task", "pcutoff") * ProjectConfig.Bodypart

### 2b. The recording: shared video and poses, the lab's arena

`register_recording` writes the session's video (`video.VideoRecording`: camera, take and the video's
fps, size and frame count) and its DeepLabCut poses (`dlc.Pose`, one row per keypoint in `Pose.Keypoint`)
to the shared tables. Then the lab's own `Recording` ties them together with what only this lab knows:
the arena, and where its corners are in the image.

The session's details and welfare check go to the lab's `SessionInfo` and `ScoreSheet`.

In [ ]:
from scene_pipelines.schemas.dlc.pose import Pose
from scene_pipelines.schemas.video.recording import VideoRecording
from scene_pipelines.schemas.video.video import VideoFile

from scene_onboarding_demo.register import register_recording, register_session_details
from scene_onboarding_demo.schemas import Recording, ScoreSheet, SessionInfo

register_session_details(manifest.session, session=session)
recording = register_recording(manifest.recording, session=session, config=config)

display(SessionInfo * ScoreSheet & session)
display((VideoRecording * VideoFile & recording).proj("fps", "width", "height", "n_frames"))
display(Pose & Recording().pose(recording))
(Recording & recording).proj("arena_name", "video_path")

### 3. Computed for you

In [ ]:
from scene_onboarding_demo.schemas import Kinematics, Trajectory

Trajectory.populate()
Kinematics.populate()

Kinematics & recording

The path from the `Trajectory` table, on a video frame warped to a top-down view of the 52 × 52 cm arena;
the dashed square is its central quarter.

In [ ]:
from scene_onboarding_demo.plots import plot_path

fig, ax = plt.subplots(figsize=(6.5, 5))
plot_path(ax, recording)
ax.set_title(manifest.session.session_code, fontsize=10);

---
## 4. The next session? One call

Months later the same mouse is tracked live with DeepLabCut-live. Everything the steps above did by hand,
`ingest` does for its folder. The mouse and the DeepLabCut project are recognised, not written again.

In [ ]:
ingest(SECOND)

In [ ]:
keys = (Recording * Session).keys(order_by="session_date")
fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 5))
for ax, key in zip(axes, keys):
    plot_path(ax, key, colorbar=ax is axes[-1])
    session_code = (Session & key).fetch1("session_code")
    scorer = (Pose & Recording().pose(key)).fetch1("scorer")
    ax.set_title(f"{session_code} ({'live' if scorer == 'DLC_live' else 'offline'})", fontsize=10)
fig.tight_layout()

Anyone with access to the database gets the same from a query, without the lab's disks: per session, the
mouse's age, its weight change at the welfare check, and how it explored. (The live session lasts 7 s
against 30 s offline, so compare with care.)

In [ ]:
per_session = (
    Session * Session.Subject * Subject * Mouse * ScoreSheet * Recording * Kinematics
).proj(
    "subject_code",
    "session_date",
    "weight_percentage",
    "distance_cm",
    "mean_speed_cm_s",
    "center_fraction",
    age_days="DATEDIFF(session_date, date_of_birth)",
)
columns = [
    "subject_code",
    "age_days",
    "session_date",
    "weight_percentage",
    "distance_cm",
    "mean_speed_cm_s",
    "center_fraction",
]
per_session.to_pandas().reset_index()[columns].round(2)

## 5. How the lab's own tables are written

The lab's tables are in
**[`schemas/behavior.py`](https://github.com/SCENE-Collaboration/SCENE-pipelines/blob/dev/examples/tutorials/scene_onboarding_demo/schemas/behavior.py)**.
They reference the shared `VideoRecording`, `dlc.Pose` and `dlc.ProjectConfig`; nothing in scene-pipelines
changes.

```text
Session ─> VideoRecording ─┐
dlc.Pose ──────────────────┼──> Recording ──> Trajectory ──> Kinematics
dlc.ProjectConfig, Arena ──┘    (Manual)      (Computed)     (Computed)
```

Look at `Kinematics`: a `definition` (what is stored per key) and a `make(key)` that reads its upstream
rows and inserts one row. That pair is all `populate()` needs to fill exactly what is missing, for this
session or the next hundred. `Trajectory.make` reads the poses back with
`scene_pipelines.ingestion.dlc.keypoint`, one keypoint at a time.

## Start over

Drops every table under this notebook's prefix; restart the kernel afterwards. Uncomment to run.

In [ ]:
# from scene_onboarding_demo.db import drop_demo_schemas

# drop_demo_schemas()